# Graph API SDK Notebook

<a id="toc" name="toc"></a>
## Table of Contents


1. [Intune adhoc samples](#td-samples)
2. [Intune Reports & Properties](#intune)
      - [Device Inventory](#device-inventory)
      - [Device Compliance](#device-compliance)
      - [Device Configuration Polices](#device-config-policies)
      - [App Management](#app-management)
      - [Endpoint Security/AV/Malware](#endpoint-security)
      - [Windows Updates](#windows-updates)
      - [Enrollment](#enrollment)
      - [Endpoint Analytics](#endpoint-analytics)
      - [Endpoint Privilege Management](#epm)
      - [Co-Management & Group Policy Analytics](#comanaged)
      - [Scripts & Remediations](#scripts)
      - [Certificates](#certs)
      - [Export Jobs](#exports)
      - [Batch Export: Common Reports](#common-reports)
  
---

<a id="samples" name="samples"></a>
## Intune adhoc samples

#### Export Settings/Policies from Source Tenant

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$ExportPath = '.\exports\SettingsCatalog'
New-Item -ItemType Directory -Force -Path $ExportPath | Out-Null

$uri      = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'
$policies = @()
do {
    $response  = Invoke-MgGraphRequest -Method GET -Uri $uri
    $policies += $response.value
    $uri       = $response.'@odata.nextLink'
} while ($uri)

Write-Host "Found $($policies.Count) Settings Catalog policies to export"

foreach ($policy in $policies) {
    $id       = $policy.id
    $full     = Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$id"
    $settings = Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$id/settings"

    $export = $full | Select-Object -ExcludeProperty id, createdDateTime, lastModifiedDateTime, settingCount
    $export | Add-Member -NotePropertyName 'settings' -NotePropertyValue $settings.value -Force
    $export | ConvertTo-Json -Depth 30 | Out-File "$ExportPath\$id.json" -Encoding utf8
    Write-Host "  Exported: $($policy.name)"
}

Write-Host "Export complete -> $ExportPath" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Just 10 polices

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$ExportPath = '.\exports\SettingsCatalog'
New-Item -ItemType Directory -Force -Path $ExportPath | Out-Null

$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies?$top=10'

$response = Invoke-MgGraphRequest -Method GET -Uri $uri
$policies = $response.value

Write-Host "Found $($policies.Count) Settings Catalog policies to export (Top 10 only)"

foreach ($policy in $policies) {
    $id = $policy.id
    $full = Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$id"
    $settings = Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$id/settings"

    $export = $full | Select-Object -ExcludeProperty id, createdDateTime, lastModifiedDateTime, settingCount
    $export | Add-Member -NotePropertyName 'settings' -NotePropertyValue $settings.value -Force
    $export | ConvertTo-Json -Depth 20 | Out-File "$ExportPath\$id.json" -Encoding utf8
    Write-Host "  Exported: $($policy.name)"
}

Write-Host ""
Write-Host "Export complete -> $ExportPath" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Validate destination tenant (before/after)

In [ ]:
# Run this before AND after the import - just compare the output
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

(Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies') | ConvertTo-Json -Depth 10

Disconnect-MgGraph | Out-Null

#### Import Settings/Policies in Destination Tenant

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$ImportPath = '.\exports\SettingsCatalog'
$jsonFiles = Get-ChildItem -Path $ImportPath -Filter '*.json'
$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'

Write-Host "Importing $($jsonFiles.Count) policies into destination tenant"

foreach ($file in $jsonFiles) {
    $raw = Get-Content $file.FullName -Raw | ConvertFrom-Json
    $settings = @($raw.settings | ForEach-Object {
            $_ | Select-Object -ExcludeProperty id
        })

    $body = @{
        name              = $raw.name
        description       = $raw.description
        platforms         = $raw.platforms
        technologies      = $raw.technologies
        templateReference = $raw.templateReference
        settings          = $settings
    }

    try {
        $result = Invoke-MgGraphRequest -Method POST `
            -Uri $uri `
            -Body ($body | ConvertTo-Json -Depth 20) `
            -ContentType 'application/json'
        Write-Host "  [OK] $($result.name) [$($result.id)]" -ForegroundColor Green
    }
    catch {
        Write-Warning "  [FAIL] $($raw.name)"
        Write-Host "    $($_.ErrorDetails.Message)" -ForegroundColor Red
        Write-Host "    $($_.Exception.Message)" -ForegroundColor Red
    }
}

Disconnect-MgGraph | Out-Null

#### Delete policy

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$policyId = 'cb4fa35a-1cac-4c7d-bc3a-6058eb98850e'

Invoke-MgGraphRequest -Method DELETE `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$policyId"

Write-Host "Policy $policyId deleted" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Delete all policies

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$policies = (Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies').value

Write-Host "Deleting $($policies.Count) policies..."

foreach ($policy in $policies) {
    Invoke-MgGraphRequest -Method DELETE `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$($policy.id)"
    Write-Host "  Deleted: $($policy.name)" -ForegroundColor Green
}

Disconnect-MgGraph | Out-Null

#### Export Legacy Config Profiles from Source Tenant

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$ExportPath = '.\exports\DeviceConfig'
New-Item -ItemType Directory -Force -Path $ExportPath | Out-Null

$uri      = 'https://graph.microsoft.com/v1.0/deviceManagement/deviceConfigurations'
$profiles = @()
do {
    $response  = Invoke-MgGraphRequest -Method GET -Uri $uri
    $profiles += $response.value
    $uri       = $response.'@odata.nextLink'
} while ($uri)

foreach ($profile in $profiles) {
    $profile | ConvertTo-Json -Depth 10 |
        Out-File "$ExportPath\$($profile.id).json" -Encoding utf8
    Write-Host "  Exported: $($profile.displayName)"
}

Write-Host "Export complete -> $ExportPath" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Validation (before/after)

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

(Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/v1.0/deviceManagement/deviceConfigurations') | ConvertTo-Json -Depth 10

Disconnect-MgGraph | Out-Null

#### Import Legacy Config Profile to Destination Tenant

In [ ]:
# Load DESTINATION tenant config
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$ImportPath = '.\exports\DeviceConfig'
$uri = 'https://graph.microsoft.com/v1.0/deviceManagement/deviceConfigurations'

foreach ($file in Get-ChildItem -Path $ImportPath -Filter '*.json') {
    $body = Get-Content $file.FullName -Raw | ConvertFrom-Json |
        Select-Object -ExcludeProperty id, createdDateTime, lastModifiedDateTime, version

    try {
        $result = Invoke-MgGraphRequest -Method POST `
            -Uri $uri `
            -Body ($body | ConvertTo-Json -Depth 10) `
            -ContentType 'application/json'
        Write-Host "  [OK] $($result.displayName)" -ForegroundColor Green
    } catch {
        Write-Warning "  [FAIL] $($file.BaseName): $($_.Exception.Message)"
    }
}

Disconnect-MgGraph | Out-Null

#### Delete all profiles

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$profiles = (Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/v1.0/deviceManagement/deviceConfigurations').value

Write-Host "Deleting $($profiles.Count) profiles..."

foreach ($profile in $profiles) {
    Invoke-MgGraphRequest -Method DELETE `
        -Uri "https://graph.microsoft.com/v1.0/deviceManagement/deviceConfigurations/$($profile.id)"
    Write-Host "  Deleted: $($profile.displayName)" -ForegroundColor Green
}

Disconnect-MgGraph | Out-Null

#### Comparing policies between tenants

In [ ]:
# ── Step 1: Collect TEST tenant policies ────────────────────────────────
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$testPolicies = @{}
$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    foreach ($p in $r.value) { $testPolicies[$p.name] = $p }
    $uri = $r.'@odata.nextLink'
} while ($uri)

Disconnect-MgGraph | Out-Null

# ── Step 2: Collect PROD tenant policies ─────────────────────────────────
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$prodPolicies = @{}
$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    foreach ($p in $r.value) { $prodPolicies[$p.name] = $p }
    $uri = $r.'@odata.nextLink'
} while ($uri)

Disconnect-MgGraph | Out-Null

# ── Report ───────────────────────────────────────────────────────────────
$inTestNotProd = $testPolicies.Keys | Where-Object { -not $prodPolicies.ContainsKey($_) }
$inProdNotTest = $prodPolicies.Keys | Where-Object { -not $testPolicies.ContainsKey($_) }

Write-Host "`nIn TEST but missing from PROD ($($inTestNotProd.Count)):" -ForegroundColor Yellow
$inTestNotProd | ForEach-Object { Write-Host "  - $_" }

Write-Host "`nIn PROD but not in TEST ($($inProdNotTest.Count)):" -ForegroundColor Cyan
$inProdNotTest | ForEach-Object { Write-Host "  - $_" }

#### Promoting Golden Policy

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$allPolicies = @()
$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'
do {
    $r            = Invoke-MgGraphRequest -Method GET -Uri $uri
    $allPolicies += $r.value
    $uri          = $r.'@odata.nextLink'
} while ($uri)

$goldenPolicies = @($allPolicies | Where-Object { $_.name -like '*MSFT*' })
Write-Host "Found $($goldenPolicies.Count) golden policies to promote"

In [ ]:
# ── Step 1: Collect GOLDEN policies from TEST tenant ─────────────────────────
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$allPolicies = @()
$uri = 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies'
do {
    $r            = Invoke-MgGraphRequest -Method GET -Uri $uri
    $allPolicies += $r.value
    $uri          = $r.'@odata.nextLink'
} while ($uri)

$goldenPolicies = @($allPolicies | Where-Object { $_.name -like '*MSFT*' })
Write-Host "Found $($goldenPolicies.Count) golden policies to promote"

$promotionQueue = foreach ($p in $goldenPolicies) {
    $settings = (Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$($p.id)/settings").value
    [pscustomobject]@{
        Name              = $p.name -replace '^GOLDEN_', ''
        Description       = "Promoted from TEST on $(Get-Date -Format 'yyyy-MM-dd')"
        Platforms         = $p.platforms
        Technologies      = $p.technologies
        TemplateReference = $p.templateReference
        Settings          = $settings
    }
}

Disconnect-MgGraph | Out-Null

In [ ]:
# ── Step 2: Import promoted policies to PROD ─────────────────────────────────
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

foreach ($policy in $promotionQueue) {
    $settings = @($policy.Settings | ForEach-Object {
        $_ | Select-Object -ExcludeProperty id
    })

    $body = @{
        name              = $policy.Name
        description       = $policy.Description
        platforms         = $policy.Platforms
        technologies      = $policy.Technologies
        templateReference = $policy.TemplateReference
        settings          = $settings
    }

    try {
        $result = Invoke-MgGraphRequest -Method POST `
            -Uri 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies' `
            -Body ($body | ConvertTo-Json -Depth 20) `
            -ContentType 'application/json'
        Write-Host "  [OK] $($result.name) [$($result.id)]" -ForegroundColor Green
    } catch {
        Write-Warning "  [FAIL] $($policy.Name)"
        Write-Host "    $($_.ErrorDetails.Message)" -ForegroundColor Red
        Write-Host "    $($_.Exception.Message)" -ForegroundColor Red
    }
}

Disconnect-MgGraph | Out-Null

#### Validate config policy (before/after)

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

(Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies').value | ConvertTo-Json -Depth 10

Disconnect-MgGraph | Out-Null

#### Delete all config policy

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$policies = (Invoke-MgGraphRequest -Method GET `
    -Uri 'https://graph.microsoft.com/beta/deviceManagement/configurationPolicies').value

Write-Host "Deleting $($policies.Count) policies..."

foreach ($policy in $policies) {
    Invoke-MgGraphRequest -Method DELETE `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/configurationPolicies/$($policy.id)"
    Write-Host "  Deleted: $($policy.name)" -ForegroundColor Green
}

Disconnect-MgGraph | Out-Null

#### List BitLocker Keys

```BitlockerKey.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

# Import-Module Microsoft.Graph.Identity.SignIns

$keys = Get-MgInformationProtectionBitlockerRecoveryKey -All
Write-Host "Total BitLocker keys in tenant: $($keys.Count)"

$keys | Select-Object Id, DeviceId, CreatedDateTime, VolumeType |
Format-Table -AutoSize

$keys | Select-Object Id, DeviceId, CreatedDateTime, VolumeType |
Export-Csv '.\reports\BitLockerKeyMetadata.csv' -NoTypeInformation -Encoding utf8
Write-Host 'Metadata exported to BitLockerKeyMetadata.csv' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Export BitLocker Keys

```BitlockerKey.Read.All```

In [ ]:
# BitLocker key value export — requires INTERACTIVE delegated session
Connect-MgGraph -Scopes 'BitlockerKey.Read.All' -NoWelcome

$allKeys = Get-MgInformationProtectionBitlockerRecoveryKey -All
Write-Host "Retrieving key values for $($allKeys.Count) keys (each creates an audit log entry)"

$report = foreach ($k in $allKeys) {
    $withKey = Get-MgInformationProtectionBitlockerRecoveryKey `
        -BitlockerRecoveryKeyId $k.Id `
        -Property 'key'
    [pscustomobject]@{
        KeyId           = $k.Id
        DeviceId        = $k.DeviceId
        CreatedDateTime = $k.CreatedDateTime
        VolumeType      = $k.VolumeType
        RecoveryKey     = $withKey.Key
    }
}

$report | Export-Csv '.\reports\BitLockerKeyValues.csv' -NoTypeInformation -Encoding utf8
Write-Host "Exported $($report.Count) keys to BitLockerKeyValues.csv" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Lookup BitLocker key by device name

```BitlockerKey.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$deviceName = 'LAPTOP-CORP-01'

# Resolve device name to Azure AD device object
$device = Get-MgDevice -Filter "displayName eq '$deviceName'" -Top 1
if (-not $device) {
    Write-Warning "Device '$deviceName' not found in Entra ID."
    Disconnect-MgGraph | Out-Null
    exit
}

# Filter BitLocker keys by deviceId
$response = Invoke-MgGraphRequest -Method GET `
    -Uri "https://graph.microsoft.com/v1.0/informationProtection/bitlocker/recoveryKeys?`$filter=deviceId eq '$($device.DeviceId)'"

if ($response.value.Count -eq 0) {
    Write-Warning "No BitLocker key found for device: $deviceName (DeviceId: $($device.DeviceId))"
}
else {
    Write-Host "BitLocker keys for $deviceName :" -ForegroundColor Cyan
    $response.value | Select-Object id, createdDateTime, volumeType | Format-Table -AutoSize
}

Disconnect-MgGraph | Out-Null

#### Device BitLocker Report

```BitlockerKey.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

# ── Collect all managed Windows devices (paginated) ──────────────────────
$selectFields = 'id,deviceName,azureADDeviceId,serialNumber,operatingSystem,osVersion,complianceState,enrolledDateTime,lastSyncDateTime'
$uri = "https://graph.microsoft.com/v1.0/deviceManagement/managedDevices?`$filter=operatingSystem eq 'Windows'&`$select=$selectFields"
$allDevices = @()
do {
    $response = Invoke-MgGraphRequest -Method GET -Uri $uri
    $allDevices += $response.value
    $uri = $response.'@odata.nextLink'
} while ($uri)
Write-Host "Managed Windows devices: $($allDevices.Count)"

# ── Collect BitLocker key metadata (no key values needed) ─────────────────
$bkKeys = Get-MgInformationProtectionBitlockerRecoveryKey -All
$keyLookup = @{}
foreach ($k in $bkKeys) { $keyLookup[$k.DeviceId] = $true }
Write-Host "BitLocker keys in tenant: $($bkKeys.Count)"

# ── Build coverage report ────────────────────────────────────────────────
$report = foreach ($d in $allDevices) {
    [pscustomobject]@{
        DeviceName       = $d.deviceName
        AzureAdDeviceId  = $d.azureADDeviceId
        SerialNumber     = $d.serialNumber
        OSVersion        = $d.osVersion
        ComplianceState  = $d.complianceState
        EnrolledDateTime = $d.enrolledDateTime
        LastSyncDateTime = $d.lastSyncDateTime
        BitLockerBacked  = $keyLookup.ContainsKey($d.azureADDeviceId)
    }
}

# ── Summary ───────────────────────────────────────────────────────────────
$backed = ($report | Where-Object { $_.BitLockerBacked }).Count
$notBacked = ($report | Where-Object { -not $_.BitLockerBacked }).Count

Write-Host "`nBitLocker Backup Summary:" -ForegroundColor Cyan
Write-Host "  Backed up   : $backed"
Write-Host "  NOT backed  : $notBacked" -ForegroundColor $(if ($notBacked -gt 0) { 'Red' } else { 'Green' })
Write-Host "  Coverage    : $([math]::Round($backed / $allDevices.Count * 100, 1))%"

# Devices without backup
if ($notBacked -gt 0) {
    Write-Host "`nDevices WITHOUT BitLocker backup:" -ForegroundColor Yellow
    $report | Where-Object { -not $_.BitLockerBacked } |
    Select-Object DeviceName, SerialNumber, OSVersion, LastSyncDateTime |
    Format-Table -AutoSize
}

# Export full report
$report | Export-Csv '.\reports\BitLockerCoverage.csv' -NoTypeInformation -Encoding utf8
Write-Host "Full report exported to BitLockerCoverage.csv" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List WHfB Registrations for user

```UserAuthenticationMethod.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

Import-Module Microsoft.Graph.Identity.SignIns

$userUpn = 'TestUser1@letsintune.com'
$whfbMethods = Get-MgUserAuthenticationWindowsHelloForBusinessMethod -UserId $userUpn -All

if ($whfbMethods.Count -eq 0) {
    Write-Host "$userUpn has NO Windows Hello for Business registrations" -ForegroundColor Yellow
}
else {
    Write-Host "$userUpn has $($whfbMethods.Count) WHfB registration(s):" -ForegroundColor Green
    $whfbMethods | Select-Object Id, DisplayName, CreatedDateTime, KeyStrength |
    Format-Table -AutoSize
}

Disconnect-MgGraph | Out-Null

#### Remove WHfB Registration for user

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration_dest.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

Import-Module Microsoft.Graph.Identity.SignIns

$userUpn = 'user@contoso.com'
$whfbKeyId = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'  # From Get- cmdlet above

Remove-MgUserAuthenticationWindowsHelloForBusinessMethod `
    -UserId $userUpn `
    -WindowsHelloForBusinessAuthenticationMethodId $whfbKeyId

Write-Host "WHfB key $whfbKeyId removed for user $userUpn" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### PasswordLess Adoption Report

```AuditLog.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

# Paginate through all user registration details
$uri = 'https://graph.microsoft.com/beta/reports/authenticationMethods/userRegistrationDetails'
$allData = @()
do {
    $response = Invoke-MgGraphRequest -Method GET -Uri $uri
    $allData += $response.value
    $uri = $response.'@odata.nextLink'
} while ($uri)

$report = $allData | ForEach-Object {
    [pscustomobject]@{
        UserPrincipalName     = $_.userPrincipalName
        DisplayName           = $_.userDisplayName
        IsAdmin               = $_.isAdmin
        IsMfaCapable          = $_.isMfaCapable
        IsPasswordlessCapable = $_.isPasswordlessCapable
        WHfB_Registered       = ($_.methodsRegistered -contains 'windowsHelloForBusiness')
        FIDO2_Registered      = ($_.methodsRegistered -contains 'passKeyDeviceBound')
        AuthApp_Registered    = ($_.methodsRegistered -contains 'microsoftAuthenticatorPasswordless')
        AllMethods            = ($_.methodsRegistered -join '; ')
    }
}

$total = $report.Count
$whfbCount = ($report | Where-Object WHfB_Registered).Count
$fido2Count = ($report | Where-Object FIDO2_Registered).Count
$noPassless = ($report | Where-Object {
        -not $_.WHfB_Registered -and -not $_.FIDO2_Registered -and -not $_.AuthApp_Registered
    }).Count

Write-Host "`nPasswordless Adoption Summary:" -ForegroundColor Cyan
Write-Host "  Total users        : $total"
Write-Host "  WHfB registered    : $whfbCount ($([math]::Round($whfbCount/$total*100,1))%)"
Write-Host "  FIDO2 registered   : $fido2Count"
Write-Host "  No passwordless    : $noPassless" -ForegroundColor $(if ($noPassless -gt 0) { 'Yellow' } else { 'Green' })

$report | Export-Csv '.\reports\WHfBAdoptionReport.csv' -NoTypeInformation -Encoding utf8
Write-Host 'Report exported to WHfBAdoptionReport.csv' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List users with no Passwordless methods registered

```AuditLog.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$uri = 'https://graph.microsoft.com/beta/reports/authenticationMethods/userRegistrationDetails'
$allData = @()
do {
    $response = Invoke-MgGraphRequest -Method GET -Uri $uri
    $allData += $response.value
    $uri = $response.'@odata.nextLink'
} while ($uri)

$noPasswordless = $allData | Where-Object {
    $_.methodsRegistered -notcontains 'windowsHelloForBusiness' -and
    $_.methodsRegistered -notcontains 'passKeyDeviceBound' -and
    $_.methodsRegistered -notcontains 'microsoftAuthenticatorPasswordless'
} | ForEach-Object {
    [pscustomobject]@{
        UserPrincipalName = $_.userPrincipalName
        DisplayName       = $_.userDisplayName
        IsAdmin           = $_.isAdmin
        MethodsRegistered = ($_.methodsRegistered -join '; ')
    }
}

Write-Host "Users with no passwordless method: $($noPasswordless.Count)" -ForegroundColor Red
$noPasswordless | Sort-Object IsAdmin -Descending | Format-Table -AutoSize

$noPasswordless | Export-Csv '.\reports\NoPasswordlessUsers.csv' -NoTypeInformation -Encoding utf8

Disconnect-MgGraph | Out-Null

#### List auth methods for user

```UserAuthenticationMethod.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

Import-Module Microsoft.Graph.Identity.SignIns

$userUpn = 'TestUser1@letsintune.com'
$methods = Get-MgUserAuthenticationMethod -UserId $userUpn -All

Write-Host "Authentication methods for $userUpn :" -ForegroundColor Cyan
foreach ($method in $methods) {
    $type = $method.AdditionalProperties['@odata.type'] -replace '#microsoft.graph.', ''
    Write-Host "  $type  (Id: $($method.Id))"
}

Disconnect-MgGraph | Out-Null

#### MFA registration report

```AuditLog.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$uri = 'https://graph.microsoft.com/beta/reports/authenticationMethods/userRegistrationDetails'
$data = @()
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    $data += $r.value
    $uri = $r.'@odata.nextLink'
} while ($uri)

$report = $data | ForEach-Object {
    [pscustomobject]@{
        UPN                 = $_.userPrincipalName
        DisplayName         = $_.userDisplayName
        IsAdmin             = $_.isAdmin
        MfaCapable          = $_.isMfaCapable
        MfaRegistered       = $_.isMfaRegistered
        SsprCapable         = $_.isSsprCapable
        SsprRegistered      = $_.isSsprRegistered
        PasswordlessCapable = $_.isPasswordlessCapable
        DefaultMfaMethod    = $_.defaultMfaMethod
        MethodsRegistered   = ($_.methodsRegistered -join '; ')
    }
}

# Summary stats
$total = $report.Count
Write-Host "`nMFA Registration Summary ($total users):" -ForegroundColor Cyan
Write-Host "  MFA Capable          : $(($report | Where MfaCapable).Count)"
Write-Host "  MFA Registered       : $(($report | Where MfaRegistered).Count)"
Write-Host "  SSPR Capable         : $(($report | Where SsprCapable).Count)"
Write-Host "  Passwordless Capable : $(($report | Where PasswordlessCapable).Count)"

$report | Export-Csv '.\reports\MfaRegistrationStatus.csv' -NoTypeInformation -Encoding utf8
Write-Host 'Report exported to MfaRegistrationStatus.csv' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Read auth method policies

```Policy.Read.AuthenticationMethod```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$uri = 'https://graph.microsoft.com/v1.0/policies/authenticationMethodsPolicy'
# Get the authentication methods policy (what methods are enabled tenant-wide)
$policy = Invoke-MgGraphRequest -Method GET -Uri $uri

Write-Host "Authentication Methods Policy:" -ForegroundColor Cyan
Write-Host "  Policy version: $($policy.policyVersion)"

foreach ($method in $policy.authenticationMethodConfigurations) {
    $state = $method.state
    $methodType = $method.'@odata.type' -replace '#microsoft.graph.', ''
    $stateColor = if ($state -eq 'enabled') { 'Green' } else { 'Gray' }
    Write-Host "  [$state] $methodType" -ForegroundColor $stateColor
}

Disconnect-MgGraph | Out-Null

#### Enable auth method for specific group

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

$groupId = 'your-entra-group-id-here'
$uri = 'https://graph.microsoft.com/v1.0/policies/authenticationMethodsPolicy/authenticationMethodConfigurations/fido2'
# Example: Enable FIDO2 passwordless for a pilot group
$body = @{
    '@odata.type'  = '#microsoft.graph.fido2AuthenticationMethodConfiguration'
    state          = 'enabled'
    includeTargets = @(@{
            targetType             = 'group'
            id                     = $groupId
            isRegistrationRequired = $false
        })
}

Invoke-MgGraphRequest -Method PATCH `
    -Uri $uri `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host 'FIDO2 authentication method enabled for group' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Query Sign-In logs filtered by auth method

```AuditLog.Read.All```

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId -TenantId $config.TenantId -CertificateThumbprint $config.Thumbprint -NoWelcome

# Retrieve sign-ins where Windows Hello for Business was used
# authenticationDetails is beta only
$filter = "createdDateTime ge $(( Get-Date).AddDays(-90).ToString('yyyy-MM-ddTHH:mm:ssZ'))"

$uri = "https://graph.microsoft.com/beta/auditLogs/signIns?`$filter=$filter&`$top=50"
$signIns = @()
do {
    $response = Invoke-MgGraphRequest -Method GET -Uri $uri
    $signIns += $response.value
    $uri = $response.'@odata.nextLink'
} while ($uri -and $signIns.Count -lt 500)

# Filter client-side for WHfB sign-ins
$whfbSignIns = $signIns | Where-Object {
    $_.authenticationDetails.authenticationMethod -contains 'Windows Hello for Business'
}

Write-Host "Total sign-ins (last 7d): $($signIns.Count)"
Write-Host "WHfB sign-ins           : $($whfbSignIns.Count)"

$whfbSignIns | Select-Object userPrincipalName, createdDateTime,
@{n = 'AppDisplayName'; e = { $_.appDisplayName } },
@{n = 'AuthMethod'; e = { $_.authenticationDetails.authenticationMethod -join '; ' } } |
Format-Table -AutoSize

$whfbSignIns | ConvertTo-Json -Depth 5 |
Out-File '.\reports\WHfBSignIns.json' -Encoding utf8
Write-Host 'WHfB sign-in data saved to WHfBSignIns.json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List and Read App Definitions

```DeviceManagementApps.Read.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# List all apps — native cmdlet handles pagination automatically
$apps = Get-MgDeviceAppManagementMobileApp -All

Write-Host "Total apps in tenant: $($apps.Count)"
$apps | Select-Object id,
@{n = 'Type'; e = { $_.'@odata.type' -replace '#microsoft.graph.', '' } },
displayName, publisher, createdDateTime |
Sort-Object Type, displayName |
Format-Table -AutoSize

$apps | ConvertTo-Json -Depth 10 | Out-File '.\exports\AllApps.json' -Encoding utf8
Write-Host 'Exported -> .\exports\AllApps.json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List Apps by Type

```DeviceManagementApps.Read.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# Common @odata.type filters:
#   win32LobApp            — Win32 .intunewin packages
#   winGetApp              — WinGet / Microsoft Store apps
#   windowsMobileMSI       — MSI line-of-business apps
#   windowsUniversalAppX   — MSIX / AppX packages
#   officeSuiteApp         — Microsoft 365 Apps
#   webApp                 — Web links

# Example: WinGet apps only
$filter = "isof('microsoft.graph.winGetApp')"
$uri    = "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps?`$filter=$filter"
$apps   = @()
do {
    $r     = Invoke-MgGraphRequest -Method GET -Uri $uri
    $apps += $r.value
    $uri   = $r.'@odata.nextLink'
} while ($uri)

Write-Host "WinGet apps: $($apps.Count)"
$apps | Select-Object id, displayName, publisher | Format-Table -AutoSize

$apps | ConvertTo-Json -Depth 10 | Out-File '.\exports\WinGetApps.json' -Encoding utf8
Write-Host 'Exported -> .\exports\WinGetApps.json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Get a Specific App by Name

```DeviceManagementApps.Read.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appName = 'Notepad++'
$apps = Get-MgDeviceAppManagementMobileApp -Filter "displayName eq '$appName'" -All

if ($apps.Count -eq 0) {
    Write-Warning "App '$appName' not found"
}
else {
    $app = $apps[0]
    Write-Host "Found: $($app.displayName) [$($app.id)]"
    Write-Host "Type : $($app.AdditionalProperties.'@odata.type' -replace '#microsoft.graph.','' )"
    Write-Host "Publisher: $($app.publisher)"
    Write-Host "Created  : $($app.createdDateTime)"
    $app | ConvertTo-Json -Depth 10 | Out-File ".\exports\App_$($app.id).json" -Encoding utf8
    Write-Host "Exported -> .\exports\App_$($app.id).json" -ForegroundColor Green
}

Disconnect-MgGraph | Out-Null

#### Create WinGet App

```DeviceManagementApps.ReadWrite.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$body = @{
    '@odata.type'         = '#microsoft.graph.winGetApp'
    displayName           = '7-Zip'
    description           = '7-Zip file archiver'
    publisher             = 'Igor Pavlov'
    packageIdentifier     = '7zip.7zip'
    installExperience     = @{
        runAsAccount = 'system'
    }
}

$result = Invoke-MgGraphRequest -Method POST `
    -Uri 'https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps' `
    -Body ($body | ConvertTo-Json -Depth 10) `
    -ContentType 'application/json'

Write-Host "Created WinGet app: $($result.displayName) [$($result.id)]" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Create WinGet App Definition - Metadata

```DeviceManagementApps.ReadWrite.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$body = @{
    '@odata.type'                  = '#microsoft.graph.win32LobApp'
    displayName                    = 'MyApp 1.0'
    description                    = 'Corporate application deployment'
    publisher                      = 'Contoso IT'
    fileName                       = 'MyApp.intunewin'
    installCommandLine             = 'setup.exe /quiet /norestart'
    uninstallCommandLine           = 'setup.exe /uninstall /quiet'
    installExperience              = @{ runAsAccount = 'system' }
    minimumSupportedWindowsRelease = '10_21H1'
    rules                          = @(
        @{
            '@odata.type'          = '#microsoft.graph.win32LobAppProductCodeRule'
            ruleType               = 'detection'
            productCode            = '{YOUR-PRODUCT-CODE-GUID}'
            productVersion         = '1.0.0'
            productVersionOperator = 'greaterThanOrEqual'
        }
    )
}

$result = Invoke-MgGraphRequest -Method POST `
    -Uri 'https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps' `
    -Body ($body | ConvertTo-Json -Depth 10) `
    -ContentType 'application/json'

Write-Host "Created Win32 app shell: $($result.displayName) [$($result.id)]" -ForegroundColor Green
Write-Host "Next: upload .intunewin content"

Disconnect-MgGraph | Out-Null

#### Win32 App Content Upload (.intunewin)

https://github.com/microsoft/mggraph-intune-samples/blob/main/LOB_Application/readme.md

#### Create a Web App (URL Link)

```DeviceManagementApps.ReadWrite.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$body = @{
    '@odata.type'     = '#microsoft.graph.webApp'
    displayName       = 'Contoso Portal'
    description       = 'Internal employee portal'
    publisher         = 'Contoso IT'
    appUrl            = 'https://portal.contoso.com'
    useManagedBrowser = $false
}

$result = Invoke-MgGraphRequest -Method POST `
    -Uri 'https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps' `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "Created web app: $($result.displayName) [$($result.id)]" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Update App Definitions
> Only include properties you want to change. @odata.type must match the original app type.

```DeviceManagementApps.ReadWrite.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'

$body = @{
    '@odata.type' = '#microsoft.graph.winGetApp'
    description   = 'Updated description - v2'
    publisher     = 'Igor Pavlov'
}

Invoke-MgGraphRequest -Method PATCH `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId" `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "App $appId updated" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Delete App Definitions

```DeviceManagementApps.ReadWrite.All```

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

$appId = 'your-app-id-here'

# Confirm before delete
$app = Get-MgDeviceAppManagementMobileApp -MobileAppId $appId
Write-Host "Deleting: $($app.displayName) [$appId]" -ForegroundColor Yellow

$app | ConvertTo-Json -Depth 10 | Out-File ".\exports\Deleted_App_$appId.json" -Encoding utf8
Write-Host "Exported backup -> .\exports\Deleted_App_$appId.json" -ForegroundColor Cyan

Remove-MgDeviceAppManagementMobileApp -MobileAppId $appId

Write-Host "Deleted: $($app.displayName)" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Delete All Apps of a specific type

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# Example: delete all WinGet apps
$apps = Get-MgDeviceAppManagementMobileApp -All `
    -Filter "isof('microsoft.graph.winGetApp')"

Write-Host "Deleting $($apps.Count) WinGet apps..."

$apps | ConvertTo-Json -Depth 10 | Out-File '.\exports\Deleted_WinGetApps.json' -Encoding utf8
Write-Host "Exported backup -> .\exports\Deleted_WinGetApps.json" -ForegroundColor Cyan
foreach ($app in $apps) {
    Remove-MgDeviceAppManagementMobileApp -MobileAppId $app.id
    Write-Host "  Deleted: $($app.displayName)" -ForegroundColor Green
}

Disconnect-MgGraph | Out-Null

#### Assign App to a group
>The /assign action replaces ALL existing assignments. Include all desired assignments in one call.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'
$groupId = 'your-entra-group-id-here'

$body = @{
    mobileAppAssignments = @(
        @{
            '@odata.type' = '#microsoft.graph.mobileAppAssignment'
            intent        = 'required'
            target        = @{
                '@odata.type' = '#microsoft.graph.groupAssignmentTarget'
                groupId       = $groupId
            }
            settings      = @{
                '@odata.type'       = '#microsoft.graph.win32LobAppAssignmentSettings'
                notifications       = 'showAll'
                restartSettings     = $null
                installTimeSettings = $null
            }
        }
    )
}

Invoke-MgGraphRequest -Method POST `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId/assign" `
    -Body ($body | ConvertTo-Json -Depth 10) `
    -ContentType 'application/json'

Write-Host "App assigned as Required to group $groupId" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Assign App to multiple groups with different intents

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'
$requiredGroup = 'group-id-required'
$availableGroup = 'group-id-available'

# One call — replaces all assignments
$body = @{
    mobileAppAssignments = @(
        @{
            '@odata.type' = '#microsoft.graph.mobileAppAssignment'
            intent        = 'required'
            target        = @{
                '@odata.type' = '#microsoft.graph.groupAssignmentTarget'
                groupId       = $requiredGroup
            }
        },
        @{
            '@odata.type' = '#microsoft.graph.mobileAppAssignment'
            intent        = 'available'
            target        = @{
                '@odata.type' = '#microsoft.graph.groupAssignmentTarget'
                groupId       = $availableGroup
            }
        }
    )
}

Invoke-MgGraphRequest -Method POST `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId/assign" `
    -Body ($body | ConvertTo-Json -Depth 10) `
    -ContentType 'application/json'

Write-Host "App assigned to both groups" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List All Assignments for an App

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'

$assignments = Get-MgDeviceAppManagementMobileAppAssignment -MobileAppId $appId -All

Write-Host "Assignments for app $appId :"
$assignments | ForEach-Object {
    $intent = $_.intent
    $groupId = $_.target.groupId
    $type = $_.target.'@odata.type' -replace '#microsoft.graph.', ''
    Write-Host "  [$intent] $type -> $groupId"
}

$assignments | ConvertTo-Json -Depth 10 | Out-File ".\exports\Assignments_$appId.json" -Encoding utf8
Write-Host "Exported -> .\exports\Assignments_$appId.json" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Remove All Assignments from an App

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'

# Posting an empty assignments array clears all assignments
$body = @{ mobileAppAssignments = @() }

Invoke-MgGraphRequest -Method POST `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId/assign" `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "All assignments removed from app $appId" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List All App COnfig Policies

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# MDM channel — native cmdlet
$mdmConfigs = Get-MgDeviceAppManagementMobileAppConfiguration -All

# MAM channel — beta endpoint only, no native cmdlet
$mamConfigs = (Invoke-MgGraphRequest -Method GET `
        -Uri 'https://graph.microsoft.com/beta/deviceAppManagement/targetedManagedAppConfigurations').value

Write-Host "MDM app configs : $($mdmConfigs.Count)"
Write-Host "MAM app configs : $($mamConfigs.Count)"

$mdmConfigs | Select-Object id, displayName, '@odata.type' | Format-Table -AutoSize

$mdmConfigs | ConvertTo-Json -Depth 10 | Out-File '.\exports\MDMAppConfigs.json' -Encoding utf8
$mamConfigs | ConvertTo-Json -Depth 10 | Out-File '.\exports\MAMAppConfigs.json' -Encoding utf8
Write-Host 'Exported -> .\exports\MDMAppConfigs.json' -ForegroundColor Green
Write-Host 'Exported -> .\exports\MAMAppConfigs.json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Create an MDM App Config Policy
> Targets enrolled devices. payloadJson contains base64-encoded key/value XML for the app.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# Build key/value XML payload
$xmlPayload = @'
<dict>
  <key>AllowScreenCapture</key>
  <string>false</string>
  <key>MaxPinRetries</key>
  <integer>5</integer>
</dict>
'@
$payloadBase64 = [Convert]::ToBase64String([Text.Encoding]::UTF8.GetBytes($xmlPayload))

$body = @{
    '@odata.type'      = '#microsoft.graph.managedDeviceMobileAppConfiguration'
    displayName        = 'Edge Browser Config'
    description        = 'Corporate Edge settings'
    targetedMobileApps = @('com.microsoft.intune.mam.managedbrowser')
    payloadJson        = $payloadBase64
}

$result = Invoke-MgGraphRequest -Method POST `
    -Uri 'https://graph.microsoft.com/v1.0/deviceAppManagement/mobileAppConfigurations' `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "Created app config: $($result.displayName) [$($result.id)]" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### List App Protection Policies

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# iOS — native cmdlet
$iosPolicies = Get-MgDeviceAppManagementiOSManagedAppProtection -All

# Android — native cmdlet
$androidPolicies = Get-MgDeviceAppManagementAndroidManagedAppProtection -All

Write-Host "iOS protection policies    : $($iosPolicies.Count)"
Write-Host "Android protection policies: $($androidPolicies.Count)"

$iosPolicies | Select-Object id, displayName, periodOfflineBeforeAccessCheck |
Format-Table -AutoSize

$iosPolicies     | ConvertTo-Json -Depth 10 | Out-File '.\exports\iOSAppProtectionPolicies.json' -Encoding utf8
$androidPolicies | ConvertTo-Json -Depth 10 | Out-File '.\exports\AndroidAppProtectionPolicies.json' -Encoding utf8
Write-Host 'Exported -> .\exports\iOSAppProtectionPolicies.json' -ForegroundColor Green
Write-Host 'Exported -> .\exports\AndroidAppProtectionPolicies.json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Create an iOS App Protection Policy
> Common settings shown. Full schema has 40+ configurable properties.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$body = @{
    '@odata.type'                           = '#microsoft.graph.iosManagedAppProtection'
    displayName                             = 'iOS Corporate MAM Policy'
    description                             = 'Standard MAM policy for iOS devices'
    periodOfflineBeforeAccessCheck          = 'PT12H'
    periodOnlineBeforeAccessCheck           = 'PT30M'
    allowedInboundDataTransferSources       = 'managedApps'
    allowedOutboundDataTransferDestinations = 'managedApps'
    allowedOutboundClipboardSharingLevel    = 'managedAppsWithPasteIn'
    organizationalCredentialsRequired       = $false
    pinRequired                             = $true
    minimumPinLength                        = 6
    simplePinBlocked                        = $true
    maximumPinRetries                       = 5
    fingerprintBlocked                      = $false
    managedBrowserToOpenLinksRequired       = $false
    saveAsBlocked                           = $true
    screenCaptureBlocked                    = $true
}

$result = Invoke-MgGraphRequest -Method POST `
    -Uri 'https://graph.microsoft.com/v1.0/deviceAppManagement/iosManagedAppProtections' `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "Created iOS protection policy: $($result.displayName) [$($result.id)]" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Assign an App Protection Policy to a group

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$policyId = 'your-ios-policy-id'
$groupId = 'your-entra-group-id'

$body = @{
    assignments = @(@{
            target = @{
                '@odata.type' = '#microsoft.graph.groupAssignmentTarget'
                groupId       = $groupId
            }
        })
}

Invoke-MgGraphRequest -Method POST `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/iosManagedAppProtections/$policyId/assign" `
    -Body ($body | ConvertTo-Json -Depth 5) `
    -ContentType 'application/json'

Write-Host "iOS protection policy assigned to group $groupId" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Install Status for a specific app across all devices
> Returns per-device install state: installed, failed, notInstalled, uninstallFailed, pendingInstall, unknown.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$appId = 'your-app-id-here'

# Get app name first
$app = Invoke-MgGraphRequest -Method GET `
    -Uri "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId"
Write-Host "Install status for: $($app.displayName)" -ForegroundColor Cyan

$uri = "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$appId/deviceStatuses"
$statuses = @()
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    $statuses += $r.value
    $uri = $r.'@odata.nextLink'
} while ($uri)

# Summary
$grouped = $statuses | Group-Object installState
Write-Host "`nInstall Summary:"
$grouped | ForEach-Object { Write-Host "  $($_.Name): $($_.Count)" }

# Show failures
$failed = $statuses | Where-Object { $_.installState -eq 'failed' }
if ($failed.Count -gt 0) {
    Write-Host "`nFailed installs:" -ForegroundColor Red
    $failed | Select-Object deviceName, userPrincipalName, errorCode, lastSyncDateTime |
    Format-Table -AutoSize
}

$statuses | Select-Object deviceName, userPrincipalName, installState, lastSyncDateTime |
Export-Csv ".\exports\AppInstallStatus_$appId.csv" -NoTypeInformation -Encoding utf8
$statuses | ConvertTo-Json -Depth 10 | Out-File ".\exports\AppInstallStatus_$appId.json" -Encoding utf8
Write-Host "Exported -> .\exports\AppInstallStatus_$appId.csv / .json" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### All App installed on a specifc device
> Returns all software detected on the device — both managed and unmanaged apps.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$deviceName = 'LAPTOP-CORP-01'

# Resolve device name to managed device ID
$device = (Invoke-MgGraphRequest -Method GET `
        -Uri "https://graph.microsoft.com/v1.0/deviceManagement/managedDevices?`$filter=deviceName eq '$deviceName'").value

if (-not $device) {
    Write-Warning "Device '$deviceName' not found"
    Disconnect-MgGraph
    exit
}
$deviceId = $device[0].id

# Get all detected apps on this device
$uri = "https://graph.microsoft.com/beta/deviceManagement/managedDevices/$deviceId/detectedApps"
$detectedApps = @()
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    $detectedApps += $r.value
    $uri = $r.'@odata.nextLink'
} while ($uri)

Write-Host "$($detectedApps.Count) apps detected on $deviceName" -ForegroundColor Cyan
$detectedApps | Select-Object displayName, version, sizeInByte |
Sort-Object displayName | Format-Table -AutoSize

$safeName = $deviceName -replace '[\\/:*?"<>|]', '_'
$detectedApps | ConvertTo-Json -Depth 10 | Out-File ".\exports\DetectedApps_$safeName.json" -Encoding utf8
$detectedApps | Select-Object displayName, version, sizeInByte |
Export-Csv ".\exports\DetectedApps_$safeName.csv" -NoTypeInformation -Encoding utf8
Write-Host "Exported -> .\exports\DetectedApps_$safeName.csv / .json" -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### Tenant-wide Detected App Inventory
> Returns all unique apps detected across all managed devices in the tenant with device count per app.

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

$uri = 'https://graph.microsoft.com/beta/deviceManagement/detectedApps'
$detectedApps = @()
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    $detectedApps += $r.value
    $uri = $r.'@odata.nextLink'
} while ($uri)

Write-Host "Unique apps detected across tenant: $($detectedApps.Count)" -ForegroundColor Cyan

$detectedApps |
Select-Object displayName, version, deviceCount, platform |
Sort-Object deviceCount -Descending |
Format-Table -AutoSize

$detectedApps |
Select-Object displayName, version, deviceCount, platform, sizeInByte |
Export-Csv '.\exports\TenantAppInventory.csv' -NoTypeInformation -Encoding utf8
$detectedApps | ConvertTo-Json -Depth 10 | Out-File '.\exports\TenantAppInventory.json' -Encoding utf8
Write-Host 'Exported -> .\exports\TenantAppInventory.csv / .json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

#### App with failed installs

In [ ]:
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
    -TenantId $config.TenantId `
    -CertificateThumbprint $config.Thumbprint `
    -NoWelcome

# Get all apps first
$uri = 'https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps'
$apps = @()
do {
    $r = Invoke-MgGraphRequest -Method GET -Uri $uri
    $apps += $r.value
    $uri = $r.'@odata.nextLink'
} while ($uri)

Write-Host "Scanning $($apps.Count) apps for failures..."

$failureReport = foreach ($app in $apps) {
    $statusUri = "https://graph.microsoft.com/v1.0/deviceAppManagement/mobileApps/$($app.id)/deviceStatuses?`$filter=installState eq 'failed'"
    $failures = (Invoke-MgGraphRequest -Method GET -Uri $statusUri).value
    foreach ($f in $failures) {
        [pscustomobject]@{
            AppName           = $app.displayName
            AppId             = $app.id
            DeviceName        = $f.deviceName
            UserPrincipalName = $f.userPrincipalName
            ErrorCode         = $f.errorCode
            LastSync          = $f.lastSyncDateTime
        }
    }
}

Write-Host "Total failed installs across tenant: $($failureReport.Count)" -ForegroundColor Red
$failureReport | Sort-Object AppName | Format-Table -AutoSize

$failureReport | Export-Csv '.\exports\AppInstallFailures.csv' -NoTypeInformation -Encoding utf8
$failureReport | ConvertTo-Json -Depth 10 | Out-File '.\exports\AppInstallFailures.json' -Encoding utf8
Write-Host 'Exported -> .\exports\AppInstallFailures.csv / .json' -ForegroundColor Green

Disconnect-MgGraph | Out-Null

[back to top](#toc)

---
<a id="intune" name="intune"></a>
## Intune Graph API – Reports & Properties

> **Reference:** [Intune Reports and Properties Available Using Graph API](https://learn.microsoft.com/en-us/intune/intune-service/fundamentals/reports-export-graph-available-reports)

This notebook demonstrates how to query and export Intune reports via Microsoft Graph API using PowerShell (`Microsoft.Graph` SDK cmdlets and `Invoke-MgGraphRequest`).

### Required App Permissions
| Permission | Purpose |
|---|---|
| `DeviceManagementManagedDevices.Read.All` | Devices, compliance, scripts |
| `DeviceManagementConfiguration.Read.All` | Configuration policies |
| `DeviceManagementApps.Read.All` | Apps, MAM |
| `DeviceManagementRBAC.Read.All` | RBAC / enrollment |
| `DeviceManagementServiceConfig.Read.All` | Enrollment configs |

### Export Jobs Pattern
The `/beta/deviceManagement/reports/exportJobs` API is **asynchronous**:
1. `POST` with a `reportName` → receive a job `id`
2. `GET` the job until `status = "completed"`
3. Download the ZIP from the `url` in the completed response

## Connect to Microsoft Graph

In [ ]:
Disconnect-MgGraph | Out-Null

#### Auth with Cert from local machine

In [ ]:
# Load config and connect
$config = Get-Content ".\config\clientconfiguration.json" -Raw | ConvertFrom-Json
Connect-MgGraph -ClientId $config.ClientId `
                -TenantId $config.TenantId `
                -CertificateThumbprint $config.Thumbprint `
                -NoWelcome

Write-Host "Connected to tenant: $((Get-MgContext).TenantId)"

## Helper Function: `Invoke-IntuneExportJob`
Run this cell once. The function is used throughout all export sections below.
It submits an async export job, polls until complete, then downloads the ZIP.

In [ ]:
function Invoke-IntuneExportJob {
    <#
    .SYNOPSIS
        Submits a report export job, polls until complete, and downloads the ZIP.
    .PARAMETER ReportName
        The reportName value (e.g., "Devices", "DeviceCompliance").
    .PARAMETER Filter
        Optional OData filter string for supported reports.
    .PARAMETER Select
        Optional array of column names to include (empty = all columns).
    .PARAMETER Format
        "csv" (default) or "json".
    .PARAMETER OutputPath
        Destination folder for the downloaded ZIP. Defaults to ".\exports".
    .PARAMETER PollIntervalSeconds
        Polling interval in seconds. Default = 10.
    #>
    [CmdletBinding()]
    param (
        [Parameter(Mandatory)] [string]  $ReportName,
        [string]   $Filter              = "",
        [string[]] $Select              = @(),
        [ValidateSet("csv","json")]
        [string]   $Format              = "csv",
        [string]   $OutputPath          = ".\exports",
        [int]      $PollIntervalSeconds = 10
    )

    $body = @{ reportName = $ReportName; format = $Format }
    if ($Filter) { $body["filter"] = $Filter }
    if ($Select)  { $body["select"] = $Select  }

    Write-Host "[$(Get-Date -f HH:mm:ss)] Submitting job: $ReportName" -ForegroundColor Cyan

    $job = Invoke-MgGraphRequest `
        -Method POST `
        -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/exportJobs" `
        -Body ($body | ConvertTo-Json -Depth 5) `
        -ContentType "application/json"

    $jobUri = "https://graph.microsoft.com/beta/deviceManagement/reports/exportJobs('$($job.id)')"

    do {
        Start-Sleep -Seconds $PollIntervalSeconds
        $status = Invoke-MgGraphRequest -Method GET -Uri $jobUri
        Write-Host "[$(Get-Date -f HH:mm:ss)]   Status: $($status.status)" -ForegroundColor DarkGray
    } until ($status.status -in @("completed","failed"))

    if ($status.status -eq "failed") {
        Write-Warning "Export job FAILED for $ReportName (JobId: $($job.id))"
        return $null
    }

    New-Item -ItemType Directory -Path $OutputPath -Force | Out-Null
    $filePath = Join-Path $OutputPath "$ReportName-$(Get-Date -f yyyyMMdd-HHmmss).zip"
    Invoke-WebRequest -Uri $status.url -OutFile $filePath -UseBasicParsing
    Write-Host "[$(Get-Date -f HH:mm:ss)] Saved: $filePath" -ForegroundColor Green
    return $filePath
}

Write-Host "Invoke-IntuneExportJob loaded." -ForegroundColor Green


---
<a id="device-inventory" name="device-inventory"></a>
## Section 1 – Device Inventory

### 1a. All Managed Devices – SDK cmdlet
Returns a lightweight device list with key inventory properties using `Get-MgDeviceManagementManagedDevice`.

In [ ]:
$devices = Get-MgDeviceManagementManagedDevice -All -Property `
    deviceName, id, operatingSystem, osVersion, complianceState,
    lastSyncDateTime, enrolledDateTime, userPrincipalName, manufacturer, model

$devices | Select-Object deviceName, operatingSystem, osVersion,
    complianceState, lastSyncDateTime, userPrincipalName |
    Sort-Object deviceName | Format-Table -AutoSize

Write-Host "Total devices: $($devices.Count)" -ForegroundColor Cyan

### 1b. Export "Devices" Report
Full device list export via the async Export Jobs API. Equivalent to **Devices > All Devices > Export** in the Intune admin center.

In [ ]:
$devicesExport = Invoke-IntuneExportJob -ReportName "Devices" -OutputPath ".\exports"

### 1c. Export "DevicesWithInventory" Report
Includes hardware and software inventory details. Equivalent to **Devices > All Devices > Export** with full columns.

In [ ]:
$devicesInvExport = Invoke-IntuneExportJob -ReportName "DevicesWithInventory" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="device-compliance" name="device-compliance"></a>
## Section 2 – Device Compliance

### 2a. Non-Compliant Devices – SDK filter
Uses an OData `$filter` on `complianceState` to return only non-compliant devices in real time.

In [ ]:
$nonCompliant = Get-MgDeviceManagementManagedDevice -All `
    -Filter "complianceState eq 'noncompliant'" `
    -Property deviceName, userPrincipalName, complianceState, osVersion, lastSyncDateTime

$nonCompliant | Select-Object deviceName, userPrincipalName, complianceState,
    osVersion, lastSyncDateTime | Format-Table -AutoSize

Write-Host "Non-compliant device count: $($nonCompliant.Count)" -ForegroundColor Yellow

### 2b. Export "DeviceCompliance" Report
Compliance org report — maps to **Reports > Device Compliance** in the admin center.

In [ ]:
$complianceExport = Invoke-IntuneExportJob -ReportName "DeviceCompliance" -OutputPath ".\exports"

### 2c. Export "DeviceNonCompliance" Report
Non-compliant device details — maps to **Reports > Device Management > Device Compliance > Reports**.

In [ ]:
$nonCompExport = Invoke-IntuneExportJob -ReportName "DeviceNonCompliance" -OutputPath ".\exports"

### 2d. Export "DevicesWithoutCompliancePolicy" Report
Devices that have no compliance policy assigned.

In [ ]:
$noPolExport = Invoke-IntuneExportJob -ReportName "DevicesWithoutCompliancePolicy" -OutputPath ".\exports"

[back to top](#toc)

---
<a id="device-config-policies" name="device-config-policies"></a>
## Section 3 – Device Configuration Policies

### 3a. List Configuration Policies – SDK cmdlet
Enumerates all device configuration profiles using `Get-MgDeviceManagementDeviceConfiguration`.

In [ ]:
$configPolicies = Get-MgDeviceManagementDeviceConfiguration -All `
    -Property id, displayName, lastModifiedDateTime, version, roleScopeTagIds

$configPolicies | Select-Object displayName, lastModifiedDateTime, version, id |
    Sort-Object displayName | Format-Table -AutoSize

Write-Host "Total configuration policies: $($configPolicies.Count)" -ForegroundColor Cyan


### 3b. Per-Device Status for a Configuration Policy – SDK cmdlet
Returns the device check-in status for a specific policy. Replace `<POLICY-ID>` with a real ID from cell 3a.

In [ ]:
$configPolicyId = "<POLICY-ID>"   # <-- replace with real policy ID

$deviceStatuses = Get-MgDeviceManagementDeviceConfigurationDeviceStatus `
    -DeviceConfigurationId $configPolicyId -All `
    -Property deviceDisplayName, userPrincipalName, status, lastReportedDateTime

$deviceStatuses | Select-Object deviceDisplayName, userPrincipalName, status,
    lastReportedDateTime | Format-Table -AutoSize


### 3c. Export "ConfigurationPolicyAggregate" Report
Policy-level summary — maps to **Devices > Manage Devices > Configuration**.

In [ ]:
$cfgAggExport = Invoke-IntuneExportJob -ReportName "ConfigurationPolicyAggregate" -OutputPath ".\exports"


### 3d. Export "DeviceStatusesByConfigurationProfile" Report (filtered by policy)
Per-device check-in statuses for a specific configuration policy.

In [ ]:
$configPolicyId = "<POLICY-ID>"   # <-- replace with real policy ID

$cfgDevExport = Invoke-IntuneExportJob `
    -ReportName "DeviceStatusesByConfigurationProfile" `
    -Filter     "PolicyId eq '$configPolicyId'" `
    -OutputPath ".\exports"


### 3e. Export "DeviceAssignmentStatusByConfigurationPolicy" Report
Assignment status per device for a given policy.

In [ ]:
$configPolicyId = "<POLICY-ID>"   # <-- replace with real policy ID

$assignExport = Invoke-IntuneExportJob `
    -ReportName "DeviceAssignmentStatusByConfigurationPolicy" `
    -Filter     "PolicyId eq '$configPolicyId'" `
    -OutputPath ".\exports"


### 3f. Export "ADMXSettingsByDeviceByPolicy" Report
ADMX/GPO-backed settings status — maps to **Reports > Device Management > Device Configuration**.

In [ ]:
$admxExport = Invoke-IntuneExportJob -ReportName "ADMXSettingsByDeviceByPolicy" -OutputPath ".\exports"


### 3g. Configuration Policy Non-Compliance Summary – Reporting POST
Returns a per-policy summary of compliant, non-compliant, and error device counts for configuration policies (Devices > Manage devices > Configuration). This covers classic Device Configuration profiles, Settings Catalog policies, Security Baselines, and ADMX templates — identified by the UnifiedPolicyType column. 

Maps to **Reports > Device management > Device Compliance > Reports > Non-compliant** policies filtered to configuration policy types.

In [ ]:
# UnifiedPolicyType reference:
#   0  = Device Configuration (classic profiles)
#   2  = Settings Catalog
#   7  = Security Baseline
#   11 = ADMX / Group Policy Analytics
#
# PolicyPlatformType reference:
#   1  = Android
#   2  = iOS / iPadOS
#   10 = Windows 10 and later
#   14 = Linux

$response = Invoke-MgGraphRequest `
    -Method POST `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/getConfigurationPolicyNonComplianceSummaryReport" `
    -Body (@{
        filter  = ""
        select  = @("PolicyId","PolicyName","PolicyPlatformType","UnifiedPolicyType",
                    "NumberOfCompliantDevices","NumberOfNonCompliantDevices","NumberOfErrorDevices")
        skip    = 0
        top     = 50
        orderBy = @("PolicyName asc")
    } | ConvertTo-Json -Depth 5) `
    -ContentType "application/json" `
    -OutputType  Hashtable

$cols = $response["schema"] | ForEach-Object { $_["column"] }
$results = $response["values"] | ForEach-Object {
    $row = [ordered]@{}
    for ($i = 0; $i -lt $cols.Count; $i++) { $row[$cols[$i]] = $_[$i] }
    [PSCustomObject]$row
}

$results | Sort-Object NumberOfNonCompliantDevices -Descending | Format-Table -AutoSize

$exportPath = ".\exports\ConfigPolicyNonComplianceSummary-$(Get-Date -f yyyyMMdd-HHmmss).csv"
$results | Export-Csv -Path $exportPath -NoTypeInformation -Encoding UTF8
Write-Host "Exported $($results.Count) rows to: $exportPath" -ForegroundColor Green

[back to top](#toc)

---
<a id="app-management" name="app-management"></a>
## Section 4 – App Management

### 4a. All Mobile Apps – SDK cmdlet
Lists all apps in the tenant using `Get-MgDeviceAppManagementMobileApp`.

In [ ]:
$apps = Get-MgDeviceAppManagementMobileApp -All `
    -Property id, displayName, publishingState, isAssigned, lastModifiedDateTime

$apps | Select-Object displayName, publishingState, isAssigned, lastModifiedDateTime, id |
    Sort-Object displayName | Format-Table -AutoSize

Write-Host "Total apps: $($apps.Count)" -ForegroundColor Cyan


### 4b. App Install Status for a Specific App – Reporting POST
Returns per-device install state for one app. Replace `<APP-ID>` with an ID from cell 4a.

In [ ]:
$appId = "<APP-ID>"   # <-- replace with real app ID

$response = Invoke-MgGraphRequest `
    -Method POST `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/getDeviceInstallStatusReport" `
    -Body (@{
        filter  = "ApplicationId eq '$appId'"
        select  = @("DeviceName","UserName","Platform","AppVersion","InstallState","LastModifiedDateTime")
        skip    = 0
        top     = 50
        orderBy = @("DeviceName asc")
    } | ConvertTo-Json -Depth 5) `
    -ContentType "application/json"

$cols = $response.schema | ForEach-Object { $_.column }
$response.values | ForEach-Object {
    $row = [ordered]@{}
    for ($i = 0; $i -lt $cols.Count; $i++) { $row[$cols[$i]] = $_[$i] }
    [PSCustomObject]$row
} | Format-Table -AutoSize


### 4c. Export "AppInstallStatusAggregate" Report
Aggregate install status across all apps — maps to **Apps > Monitor > App install status**.

In [ ]:
$appInstExport = Invoke-IntuneExportJob -ReportName "AppInstallStatusAggregate" -OutputPath ".\exports"


### 4d. Export "AllAppsList" Report
Full app inventory export — maps to **Apps > All Apps**.

In [ ]:
$allAppsExport = Invoke-IntuneExportJob -ReportName "AllAppsList" -OutputPath ".\exports"


### 4e. Export "AppInvAggregate" Report – Discovered Apps
Aggregated discovered app inventory — maps to **Apps > Monitor > Discovered apps**.

In [ ]:
$appInvExport = Invoke-IntuneExportJob -ReportName "AppInvAggregate" -OutputPath ".\exports"


### 4f. Export "AppInvByDevice" Report – Discovered Apps per Device
Per-device discovered app list — maps to **Devices > All Devices > {Device} > Discovered Apps**.

In [ ]:
$appInvDevExport = Invoke-IntuneExportJob -ReportName "AppInvByDevice" -OutputPath ".\exports"


### 4g. Export "MAMAppProtectionStatus" Report
MAM app protection report — maps to **Apps > Monitor > App protection status**.

In [ ]:
$mamProtExport = Invoke-IntuneExportJob -ReportName "MAMAppProtectionStatus" -OutputPath ".\exports"


### 4h. Export "MAMAppConfigurationStatus" Report
MAM app configuration report — maps to **Apps > Monitor > App protection status > App configuration report**.

In [ ]:
$mamCfgExport = Invoke-IntuneExportJob -ReportName "MAMAppConfigurationStatus" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="endpoint-security" name="endpoint-security"></a>
## Section 5 – Endpoint Security / Antivirus / Malware

### 5a. Export "ActiveMalware" Report
Currently active malware detections — maps to **Endpoint Security > Antivirus > Win10 detected malware**.

In [ ]:
$activeMalwareExport = Invoke-IntuneExportJob -ReportName "ActiveMalware" -OutputPath ".\exports"


### 5b. Export "Malware" Report
All detected malware — maps to **Reports > Microsoft Defender > Detected malware**.

In [ ]:
$malwareExport = Invoke-IntuneExportJob -ReportName "Malware" -OutputPath ".\exports"


### 5c. Export "DefenderAgents" Report
Defender agent status per device — maps to **Reports > Microsoft Defender > Agent Status**.

In [ ]:
$defenderExport = Invoke-IntuneExportJob -ReportName "DefenderAgents" -OutputPath ".\exports"


### 5d. Export "FirewallStatus" Report
MDM firewall status for Windows 10+ — maps to **Reports > Firewall > MDM Firewall status**.

In [ ]:
$firewallExport = Invoke-IntuneExportJob -ReportName "FirewallStatus" -OutputPath ".\exports"


### 5e. Export "FirewallUnhealthyStatus" Report
Devices with unhealthy firewall configuration — maps to **Reports > Endpoint Security > Firewall**.

In [ ]:
$firewallUnhealthyExport = Invoke-IntuneExportJob -ReportName "FirewallUnhealthyStatus" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="windows-updates" name="windows-updates"></a>
## Section 6 – Windows Updates

### 6a. Export "FeatureUpdatePolicyStatusSummary" Report
Summary of feature update policy status — maps to **Devices > Manage updates > Windows updates > Feature updates**.

In [ ]:
$featureUpdSummaryExport = Invoke-IntuneExportJob -ReportName "FeatureUpdatePolicyStatusSummary" -OutputPath ".\exports"


### 6b. Export "FeatureUpdateDeviceState" Report
Per-device feature update state — maps to **Reports > Windows Updates > Windows Feature Update Report**.

In [ ]:
$featureUpdDevExport = Invoke-IntuneExportJob -ReportName "FeatureUpdateDeviceState" -OutputPath ".\exports"


### 6c. Export "FeatureUpdatePolicyFailuresAggregate" Report
Aggregate feature update failures — maps to **Devices > Monitor > Failure for feature updates**.

In [ ]:
$featureFailExport = Invoke-IntuneExportJob -ReportName "FeatureUpdatePolicyFailuresAggregate" -OutputPath ".\exports"


### 6d. Export "DriverUpdatePolicyStatusSummary" Report
Driver update policy status — maps to **Devices > Manage updates > Windows updates > Driver updates**.

In [ ]:
$driverUpdExport = Invoke-IntuneExportJob -ReportName "DriverUpdatePolicyStatusSummary" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="enrollment" name="enrollment"></a>
## Section 7 – Enrollment

### 7a. Export "DeviceEnrollmentFailures" Report
Enrollment failure details — maps to **Devices > Device onboarding > Enrollment > Monitor**.

In [ ]:
$enrollFailExport = Invoke-IntuneExportJob -ReportName "DeviceEnrollmentFailures" -OutputPath ".\exports"


### 7b. Export "EnrollmentActivity" Report
Enrollment activity summary — maps to **Dashboard > Device enrollment**.

In [ ]:
$enrollActivityExport = Invoke-IntuneExportJob -ReportName "EnrollmentActivity" -OutputPath ".\exports"


### 7c. Export "AutopilotV2DeploymentStatus" Report
Windows Autopilot deployment status — maps to **Devices > Windows > Enrollment > Windows Autopilot**.

In [ ]:
$autopilotExport = Invoke-IntuneExportJob -ReportName "AutopilotV2DeploymentStatus" -OutputPath ".\exports"


### 7d. Enrollment Failure Details – Reporting POST
Returns enrollment failure records with failure category and reason columns.

In [ ]:
$response = Invoke-MgGraphRequest `
    -Method POST `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/getEnrollmentFailureReport" `
    -Body (@{
        filter  = ""
        select  = @("DeviceName","UserPrincipalName","Os","FailureCategory","FailureReason","EnrollmentStartDate")
        skip    = 0
        top     = 50
        orderBy = @("EnrollmentStartDate desc")
    } | ConvertTo-Json -Depth 5) `
    -ContentType "application/json"

$cols = $response.schema | ForEach-Object { $_.column }
$response.values | ForEach-Object {
    $row = [ordered]@{}
    for ($i = 0; $i -lt $cols.Count; $i++) { $row[$cols[$i]] = $_[$i] }
    [PSCustomObject]$row
} | Format-Table -AutoSize


[back to top](#toc)

---
<a id="endpoint-analytics" name="endpoint-analytics"></a>
## Section 8 – Endpoint Analytics

### 8a. Export "EAStartupPerfDevicePerformance" Report
Startup performance per device — maps to **Reports > Endpoint analytics > Startup performance > Device performance**.

In [ ]:
$eaStartupDevExport = Invoke-IntuneExportJob -ReportName "EAStartupPerfDevicePerformance" -OutputPath ".\exports"


### 8b. Export "EAStartupPerfModelPerformance" Report
Startup performance per model — maps to **Reports > Endpoint analytics > Startup performance > Model performance**.

In [ ]:
$eaStartupModelExport = Invoke-IntuneExportJob -ReportName "EAStartupPerfModelPerformance" -OutputPath ".\exports"


### 8c. Export "EAAppPerformance" Report
Application reliability — maps to **Reports > Endpoint analytics > Application reliability > App performance**.

In [ ]:
$eaAppPerfExport = Invoke-IntuneExportJob -ReportName "EAAppPerformance" -OutputPath ".\exports"


### 8d. Export "EAWFADeviceList" Report
Work from anywhere device list — maps to **Reports > Endpoint analytics > Work from anywhere**.

In [ ]:
$eaWFAExport = Invoke-IntuneExportJob -ReportName "EAWFADeviceList" -OutputPath ".\exports"


### 8e. Export "BRDeviceBatteryAgg" Report
Battery health per device — maps to **Reports > Endpoint analytics > Battery health**.

In [ ]:
$batteryExport = Invoke-IntuneExportJob -ReportName "BRDeviceBatteryAgg" -OutputPath ".\exports"


### 8f. Export "EAResourcePerfAggByDevice" Report
Resource performance (CPU/RAM) per device — maps to **Reports > Endpoint analytics > Resource performance**.

In [ ]:
$eaResourceExport = Invoke-IntuneExportJob -ReportName "EAResourcePerfAggByDevice" -OutputPath ".\exports"


### 8g. Export "EADeviceScoresV2" Report
Endpoint analytics device scores — maps to **Reports > Endpoint analytics > Device scores**.

In [ ]:
$eaScoresExport = Invoke-IntuneExportJob -ReportName "EADeviceScoresV2" -OutputPath ".\exports"


### 8h. Export "EAAnomalyAsset" Report
Anomaly detection results — maps to **Reports > Endpoint analytics > Anomalies**.

In [ ]:
$eaAnomalyExport = Invoke-IntuneExportJob -ReportName "EAAnomalyAsset" -OutputPath ".\exports"


### 8i. Export "DeviceRunStatesByProactiveRemediation" Report (filtered)
Device run states for a specific proactive remediation script.

In [ ]:
$remediationId = "<REMEDIATION-SCRIPT-ID>"   # <-- replace with real remediation ID

$proactiveExport = Invoke-IntuneExportJob `
    -ReportName "DeviceRunStatesByProactiveRemediation" `
    -Filter     "PolicyId eq '$remediationId'" `
    -OutputPath ".\exports"


[back to top](#toc)

---
<a id="epm" name="epm"></a>
## Section 9 – Endpoint Privilege Management (EPM)

### 9a. Export "EpmElevationReportElevationEvent" Report
All elevation events — maps to **Endpoint security > Endpoint Privilege Management > Elevation report**.

In [ ]:
$epmElevExport = Invoke-IntuneExportJob -ReportName "EpmElevationReportElevationEvent" -OutputPath ".\exports"


### 9b. Export "EpmAggregationReportByApplication" Report
Elevations aggregated by application.

In [ ]:
$epmByAppExport = Invoke-IntuneExportJob -ReportName "EpmAggregationReportByApplication" -OutputPath ".\exports"


### 9c. Export "EpmAggregationReportByUser" Report
Elevations aggregated by user.

In [ ]:
$epmByUserExport = Invoke-IntuneExportJob -ReportName "EpmAggregationReportByUser" -OutputPath ".\exports"


### 9d. Export "EpmAggregationReportByPublisher" Report
Elevations aggregated by publisher.

In [ ]:
$epmByPubExport = Invoke-IntuneExportJob -ReportName "EpmAggregationReportByPublisher" -OutputPath ".\exports"


### 9e. Export "EpmDeniedReport" Report
Denied elevation requests — maps to **Endpoint security > Endpoint Privilege Management > Elevation report**.

In [ ]:
$epmDeniedExport = Invoke-IntuneExportJob -ReportName "EpmDeniedReport" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="comanaged" name="comanaged"></a>
## Section 10 – Co-Management & Group Policy Analytics

### 10a. Export "ComanagedDeviceWorkloads" Report
Workload split for co-managed devices — maps to **Reports > Cloud attached devices > Co-Managed Workloads**.

In [ ]:
$coMgmtExport = Invoke-IntuneExportJob -ReportName "ComanagedDeviceWorkloads" -OutputPath ".\exports"


### 10b. Export "ComanagementEligibilityTenantAttachedDevices" Report
Devices eligible for co-management — maps to **Reports > Cloud attached devices > Co-Management Eligibility**.

In [ ]:
$coEligExport = Invoke-IntuneExportJob -ReportName "ComanagementEligibilityTenantAttachedDevices" -OutputPath ".\exports"


### 10c. Export "GPAnalyticsSettingMigrationReadiness" Report
Group policy migration readiness — maps to **Reports > Group policy analytics > Group policy migration readiness**.

In [ ]:
$gpAnalyticsExport = Invoke-IntuneExportJob -ReportName "GPAnalyticsSettingMigrationReadiness" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="scripts" name="scripts"></a>
## Section 11 – Scripts & Remediations

### 11a. List All Platform Scripts – SDK cmdlet
Enumerates all PowerShell/shell scripts deployed via Intune using `Get-MgDeviceManagementDeviceManagementScript`.

In [ ]:
$scripts = Get-MgDeviceManagementDeviceManagementScript -All `
    -Property id, displayName, lastModifiedDateTime, runAsAccount, enforceSignatureCheck

$scripts | Select-Object displayName, runAsAccount, enforceSignatureCheck,
    lastModifiedDateTime, id | Sort-Object displayName | Format-Table -AutoSize


### 11b. Device Run States for a Specific Script – SDK cmdlet
Per-device run results for one script. Replace `<SCRIPT-ID>` with an ID from cell 11a.

In [ ]:
$scriptId = "<SCRIPT-ID>"   # <-- replace with real script ID

$runStates = Get-MgDeviceManagementDeviceManagementScriptDeviceRunState `
    -DeviceManagementScriptId $scriptId -All `
    -Property runState, resultMessage, errorCode, errorDescription, lastStateUpdateDateTime

$runStates | Select-Object runState, resultMessage, errorCode,
    errorDescription, lastStateUpdateDateTime | Format-Table -AutoSize


### 11c. Export "DeviceRunStatesByScript" Report (filtered by script)
Full device run state export for one script via the Export Jobs API.

In [ ]:
$scriptId = "<SCRIPT-ID>"   # <-- replace with real script ID

$scriptRunExport = Invoke-IntuneExportJob `
    -ReportName "DeviceRunStatesByScript" `
    -Filter     "ScriptId eq '$scriptId'" `
    -OutputPath ".\exports"

[back to top](#toc)

---
<a id="certs" name="certs"></a>
## Section 12 – Certificates

### 12a. Export "AllDeviceCertificates" Report
All device certificates — maps to **Devices > Monitor > Certificates**.

In [ ]:
$certExport = Invoke-IntuneExportJob -ReportName "AllDeviceCertificates" -OutputPath ".\exports"


### 12b. Export "CertificatesByRAPolicy" Report
Certificates grouped by NDES/SCEP/PKCS RA policy.

In [ ]:
$certRaExport = Invoke-IntuneExportJob -ReportName "CertificatesByRAPolicy" -OutputPath ".\exports"


[back to top](#toc)

---
<a id="exports" name="exports"></a>
## Section 13 – Utility: Inspect Export Jobs

### 13a. List All Export Jobs
Shows all recent export jobs and their status for the tenant.

In [ ]:
$allJobs = Invoke-MgGraphRequest `
    -Method GET `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/exportJobs"

$allJobs.value |
    Select-Object id, reportName, status, requestDateTime, expirationDateTime |
    Sort-Object requestDateTime -Descending |
    Format-Table -AutoSize


### 13b. Poll a Specific Export Job by ID
Useful to check the state of a previously submitted job and retrieve the download URL.

In [ ]:
$jobId = "<JOB-ID>"   # <-- replace with a job ID from cell 13a

$jobStatus = Invoke-MgGraphRequest `
    -Method GET `
    -Uri "https://graph.microsoft.com/beta/deviceManagement/reports/exportJobs('$jobId')"

$jobStatus | Select-Object id, reportName, status, requestDateTime, url | Format-List


[back to top](#toc)

---
<a id="common-reports" name="common-reports"></a>
## Section 14 – Batch Export: Common Reports

### 14a. Batch Export Core Reports
Submits all jobs sequentially. For parallel execution, wrap each call in a `Start-Job` or `ForEach-Object -Parallel` (PS 7+).

In [ ]:
$batchReports = @(
    "Devices",
    "DeviceCompliance",
    "DeviceNonCompliance",
    "DevicesWithoutCompliancePolicy",
    "AllAppsList",
    "AppInstallStatusAggregate",
    "AppInvAggregate",
    "ActiveMalware",
    "Malware",
    "FirewallStatus",
    "DefenderAgents",
    "FeatureUpdatePolicyStatusSummary",
    "EnrollmentActivity",
    "EAStartupPerfDevicePerformance",
    "EADeviceScoresV2",
    "BRDeviceBatteryAgg"
)

New-Item -ItemType Directory -Path ".\exports\batch" -Force | Out-Null
$results = [System.Collections.Generic.List[PSCustomObject]]::new()

foreach ($report in $batchReports) {
    try {
        $path = Invoke-IntuneExportJob -ReportName $report -OutputPath ".\exports\batch"
        $results.Add([PSCustomObject]@{ Report = $report; Status = "OK"; Path = $path })
    }
    catch {
        $results.Add([PSCustomObject]@{ Report = $report; Status = "FAILED"; Path = $_.Exception.Message })
        Write-Warning "Failed: $report — $_"
    }
}

$results | Format-Table -AutoSize


[back to top](#toc)